# Day 6 · Lab 4: LangChain vs LangGraph Foundations
### Advanced RAG Curriculum · Enterprise AI Engineering · Udaan Learning

---

## 🎯 Learning Objectives
1. Understand LangChain as component toolkit and LangGraph as stateful orchestrator.
2. Build a live executable **LangGraph StateGraph** with nodes, edges, and conditional routing.



In [ ]:
from typing import TypedDict, List, Dict, Any, Optional, Annotated, Sequence
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.documents import Document

print("[*] Successfully imported LangGraph StateGraph, START, END, and LangChain Document!")

class RAGState(TypedDict):
    question: str
    route: str
    documents: Annotated[Sequence[Document], operator.add]
    evidence_sufficient: bool
    retry_count: int
    max_retries: int
    answer: Optional[str]

def understand_query(state: RAGState) -> Dict[str, Any]:
    q = state["question"]
    route = "sql" if "rahul" in q.lower() or "mrn" in q.lower() else "vector"
    return {"route": route}

def retrieve(state: RAGState) -> Dict[str, Any]:
    if state["route"] == "sql":
        docs = [Document(page_content="EHR Record: Patient Rahul Sharma | Dr. Smith | Metformin 500mg BID", metadata={"src": "sql"})]
    else:
        docs = [Document(page_content="General hospital policy on diabetes monitoring", metadata={"src": "vector"})]
    return {"documents": docs}

def evaluate_evidence(state: RAGState) -> Dict[str, Any]:
    docs_text = " ".join([d.page_content for d in state["documents"]])
    sufficient = ("Metformin" in docs_text and "Rahul" in docs_text)
    return {"evidence_sufficient": sufficient}

def generate_answer(state: RAGState) -> Dict[str, Any]:
    doc = state["documents"][0]
    return {"answer": f"Grounded Answer: {doc.page_content}"}

def rewrite_query(state: RAGState) -> Dict[str, Any]:
    return {"route": "sql", "retry_count": state.get("retry_count", 0) + 1}

def abstain(state: RAGState) -> Dict[str, Any]:
    return {"answer": "Safe Medical Abstention: Unable to verify prescription within retry budget."}

workflow = StateGraph(RAGState)
workflow.add_node("understand_query", understand_query)
workflow.add_node("retrieve", retrieve)
workflow.add_node("evaluate_evidence", evaluate_evidence)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("abstain", abstain)

workflow.add_edge(START, "understand_query")
workflow.add_edge("understand_query", "retrieve")
workflow.add_edge("retrieve", "evaluate_evidence")
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_edge("generate_answer", END)
workflow.add_edge("abstain", END)

def decide_route(state: RAGState) -> str:
    if state["evidence_sufficient"]:
        return "generate"
    elif state["retry_count"] < state["max_retries"]:
        return "retry"
    else:
        return "abstain"

workflow.add_conditional_edges("evaluate_evidence", decide_route, {
    "generate": "generate_answer",
    "retry": "rewrite_query",
    "abstain": "abstain"
})

app = workflow.compile()
print("[+] LangGraph compiled!")

res = app.invoke({
    "question": "What did Dr. Smith prescribe to Rahul?",
    "route": "",
    "documents": [],
    "evidence_sufficient": False,
    "retry_count": 0,
    "max_retries": 2,
    "answer": None
})

print("Result Answer:", res["answer"])
